# Lab 09-02 — Recursive tree build with LLM summarization (RAPTOR step 2)

**Track 09 · RAPTOR** — replacing flat chunk lists with a summary tree.

Lab 01 produced a flat partition of chunks. This lab turns it into the artifact RAPTOR actually queries: a **tree** where the leaves are the raw chunks and every internal node is a short LLM summary of the cluster of chunks beneath it.

This notebook is **self-contained**: it imports LangChain (the BGE embedder and the local Ollama LLM), pandas, and scikit-learn directly — no repo component library. The RAPTOR machinery the repo ships as `src/tools/raptor.py` — recursive GMM clustering, LLM summarization, and the recursive tree build — is hand-rolled right here, cell by cell, which is exactly how that shared component works underneath.

```text
24 chunks (rag-mini-wikipedia, deterministic head)
  -> BGE embeddings (BAAI/bge-base-en-v1.5, local, CPU)
  -> recursive GaussianMixture clustering
  -> one ChatOllama summary per cluster (qwen2.5-coder:7b, local, temperature 0)
  -> recurse on the summaries until one root remains
  -> verification gate
```

The build: level 0 is every chunk as a leaf node carrying its own text; cluster the leaf embeddings with the recursive GMM from lab 01; ask the LLM to summarize each cluster ("Summarize the following passages in 2-3 sentences, keeping key facts and names.") — that summary becomes a parent node whose children are the cluster's leaves; then recurse on the summaries until one node (the root) remains.

Two properties make the tree useful instead of a glorified flat list:

1. **Coverage** — every chunk appears in exactly one leaf, so no information is dropped while building the tree.
2. **Compression** — a query can navigate the few summary nodes at the top and only descend into the cluster that actually matters (lab 03 tests that traversal). The whole build costs one LLM call per internal node, far cheaper than re-reading every chunk at query time.


## Setup

Two prerequisites must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` (3200 passages), already fetched by the repo's manifest-verified fetchers.
- **Ollama serving `qwen2.5-coder:7b`** — the fully local LLM that writes the summaries (`ollama pull qwen2.5-coder:7b`, then `ollama serve`). No API key.

No repo imports are needed: everything this notebook uses comes from `langchain-huggingface`, `langchain-ollama`, `pandas`, `numpy`, and `scikit-learn`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   langchain-huggingface -> HuggingFaceEmbeddings (BGE backend)
#   langchain-ollama      -> ChatOllama (local qwen2.5-coder:7b summaries)
#   pandas                -> read the rag-mini-wikipedia parquet
#   scikit-learn, numpy   -> GaussianMixture clustering (hand-rolled RAPTOR)
%pip install -q langchain-huggingface langchain-ollama pandas scikit-learn


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import sys
import time
import urllib.request
from pathlib import Path

# LangChain + pandas + scikit-learn — the only libraries this notebook
# needs. Nothing is imported from the repo's src/ component library.
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from langchain_ollama import ChatOllama  # noqa: E402
from sklearn.mixture import GaussianMixture  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `PASSAGES_PATH` points at the corpus already on disk; `N_PASSAGES = 24` takes a deterministic head (the tree costs one LLM call per cluster); `MAX_CLUSTER_SIZE = 8` caps how many chunks a single summary node may cover; `BGE_MODEL_NAME` / `BGE_DEVICE` pin the embedder to the local BGE model on CPU. The LLM is `ChatOllama` with the repo's default local model `qwen2.5-coder:7b` at `temperature = 0.0` (deterministic summaries) served at `http://localhost:11434`. `ROOT_PREVIEW` / `MID_PREVIEW` truncate the summary previews the demo prints.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 24  # deterministic head; the tree costs one LLM call per cluster
MAX_CLUSTER_SIZE = 8  # max chunks a single summary node may cover
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM
OLLAMA_MODEL = "qwen2.5-coder:7b"  # local LLM, same default the repo uses
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_TEMPERATURE = 0.0  # deterministic summaries
ROOT_PREVIEW = 200  # characters of the root summary to print
MID_PREVIEW = 160  # characters of the mid-level summary to print


## 2. Load — first N passages of the rag-mini-wikipedia corpus

`load_passages` reads the first `n` rows of `passages.parquet` and returns the stripped passage texts in file order — a deterministic head, exactly the slice the lab script uses.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N passages of the rag-mini-wikipedia corpus
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


## 3. Experiment — embed, cluster, summarize recursively until one root remains

The whole pipeline is built inline, in three hand-rolled pieces. **Clustering** — `cluster_embeddings` / `_split_indices` implement the same recursive 2-component full-covariance `GaussianMixture` from lab 01 (same `max_cluster_size` cap, same fallback that keeps coverage on fit failure). **Summarization** — `_summarize` joins the cluster's passages as a bullet list behind the paper's instruction ("Summarize the following passages in 2-3 sentences, keeping key facts and names.") and hands it to `ChatOllama`; `_StubLLM` is a deterministic fallback that keeps the notebook runnable offline. **Tree build** — `build_tree` starts from one leaf node per chunk, clusters each level's node texts, makes one summary parent per cluster, and recurses until a single root remains (with a degenerate-split guard so the recursion always terminates). `_walk`, `collect_chunk_ids`, and `_node_counts_per_level` are the traversal helpers the demo and gate use.

The LLM contract is explicit: if Ollama answers at `OLLAMA_BASE_URL` the cell prints `[RUN]` and uses `ChatOllama`; if the server is down it prints `[SKIP]` and uses the deterministic stub — so the experiment and its gate pass either way, exactly like the run/skip contract the labs use for LLM-backed sections.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed, cluster, summarize recursively until one root remains
# --------------------------------------------------------------------------
SUMMARY_PROMPT = (
    "Summarize the following passages in 2-3 sentences, keeping key facts "
    "and names."
)


def _split_indices(
    index_set: list[int],
    embeddings: list[list[float]],
    max_cluster_size: int,
    seed: int,
) -> list[list[int]]:
    """Recursively split ``index_set`` into clusters of at most ``max_cluster_size``.

    Fits a 2-component full-covariance GaussianMixture on the current index
    set. A cluster small enough is kept as a leaf; anything larger is split
    again. When the fit fails (too few points, degenerate covariance, or a
    collapsed component) the whole set is returned as one cluster, so the
    partition always covers every index exactly once.
    """
    if len(index_set) <= max_cluster_size:
        return [sorted(index_set)]
    matrix = np.asarray([embeddings[i] for i in index_set], dtype=float)
    try:
        labels = GaussianMixture(
            n_components=2,
            covariance_type="full",
            random_state=seed,
        ).fit_predict(matrix)
    except (ValueError, np.linalg.LinAlgError):
        # Fallback: keep the whole set as one cluster on fit failure
        # (ill-defined covariance, n_components > n_samples) so coverage holds.
        return [sorted(index_set)]

    groups: dict[int, list[int]] = {}
    for idx, label in zip(index_set, labels):
        groups.setdefault(int(label), []).append(idx)
    if len(groups) < 2:  # degenerate split: every point in one component
        return [sorted(index_set)]

    return [
        cluster
        for group in groups.values()
        for cluster in _split_indices(group, embeddings, max_cluster_size, seed)
    ]


def cluster_embeddings(
    embeddings: list[list[float]],
    max_cluster_size: int,
    seed: int = 42,
) -> list[list[int]]:
    """Partition chunk indices into GMM clusters of at most ``max_cluster_size``."""
    if not embeddings:
        return []
    return _split_indices(
        list(range(len(embeddings))), embeddings, max_cluster_size, seed
    )


def _summarize(llm, texts: list[str]) -> str:
    """One LLM call: compress ``texts`` into 2-3 sentences of prose."""
    joined = "\n".join(f"- {text}" for text in texts)
    out = llm.invoke(f"{SUMMARY_PROMPT}\n\n{joined}")
    if hasattr(out, "content"):  # ChatOllama returns an AIMessage
        out = out.content
    return str(out).strip()


class _StubLLM:
    """Deterministic summarizer used only when Ollama is unreachable."""

    def invoke(self, prompt: str) -> str:
        lines = [line[2:] for line in prompt.splitlines() if line.startswith("- ")]
        return " ".join(line[:24] for line in lines)[:200] or "summary"


def _ollama_reachable(url: str = OLLAMA_BASE_URL, timeout: float = 2.0) -> bool:
    """True if the local Ollama server answers /api/tags."""
    try:
        with urllib.request.urlopen(f"{url}/api/tags", timeout=timeout) as resp:
            return resp.status == 200
    except Exception:
        return False


def _walk(node: dict):
    """Depth-first iterator over every node of the tree."""
    yield node
    for child in node["children"]:
        yield from _walk(child)


def collect_chunk_ids(node: dict) -> list[int]:
    """All chunk ids under ``node`` (leaf walk) — useful for gate checks."""
    if not node["children"]:
        return list(node["chunk_ids"])
    ids: list[int] = []
    for child in node["children"]:
        ids.extend(collect_chunk_ids(child))
    return ids


def _node_counts_per_level(root: dict) -> dict[int, int]:
    """Number of nodes per level (level 0 = leaves)."""
    counts: dict[int, int] = {}
    for node in _walk(root):
        counts[node["level"]] = counts.get(node["level"], 0) + 1
    return dict(sorted(counts.items()))


def build_tree(
    chunks: list[str],
    embedder,
    llm,
    max_cluster_size: int = 8,
    progress=None,
) -> dict:
    """Build the RAPTOR tree: leaves are chunks, parents are LLM summaries.

    Returns ``{"tree", "levels", "leaves", "llm_calls"}``. ``tree`` is a
    nested dict with shape ``{"text", "chunk_ids", "children", "level"}`` —
    level-0 nodes are leaves (one chunk each, empty ``children``); higher
    levels are summaries whose ``chunk_ids`` cover the whole subtree. Every
    chunk appears in exactly one leaf.
    """
    nodes = [
        {"text": chunks[i], "chunk_ids": [i], "children": [], "level": 0}
        for i in range(len(chunks))
    ]
    llm_calls = 0
    level = 0
    while len(nodes) > 1:
        level += 1
        vectors = embedder.embed_documents([node["text"] for node in nodes])
        clusters = cluster_embeddings(vectors, max_cluster_size)
        if len(clusters) == len(nodes):
            # No merge happened (degenerate GMM split): collapse to a single
            # root so the recursion always terminates.
            clusters = [list(range(len(nodes)))]
        parents: list[dict] = []
        for cluster in clusters:
            members = [nodes[i] for i in cluster]
            parents.append(
                {
                    "text": _summarize(llm, [node["text"] for node in members]),
                    "chunk_ids": sorted(
                        cid for node in members for cid in node["chunk_ids"]
                    ),
                    "children": members,
                    "level": level,
                }
            )
            llm_calls += 1
        nodes = parents
        if progress is not None:
            progress(level, len(nodes))

    if not nodes:  # empty chunk list: keep a harmless empty root
        nodes = [{"text": "", "chunk_ids": [], "children": [], "level": 0}]
    root = nodes[0]
    return {
        "tree": root,
        "levels": level,
        "leaves": len(collect_chunk_ids(root)),
        "llm_calls": llm_calls,
    }


def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": BGE_DEVICE},
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )

    # Run/skip contract: real ChatOllama when the server answers, deterministic
    # stub otherwise — the gate passes either way.
    if _ollama_reachable():
        print(f"[RUN] Ollama LLM {OLLAMA_MODEL} reachable at {OLLAMA_BASE_URL} "
              f"(temperature {OLLAMA_TEMPERATURE})")
        llm = ChatOllama(
            model=OLLAMA_MODEL,
            temperature=OLLAMA_TEMPERATURE,
            base_url=OLLAMA_BASE_URL,
        )
        llm_mode = "ollama"
    else:
        print(f"[SKIP] Ollama not reachable at {OLLAMA_BASE_URL} — using "
              "deterministic stub summarizer")
        llm = _StubLLM()
        llm_mode = "stub"

    t0 = time.perf_counter()
    info = build_tree(
        passages,
        embedder,
        llm,
        max_cluster_size=MAX_CLUSTER_SIZE,
        progress=lambda level, count: print(
            f"  built level {level}: {count} node(s)", end="\r", flush=True
        ),
    )
    build_s = time.perf_counter() - t0
    print(" " * 40, end="\r")

    mid_nodes = [
        node
        for node in _walk(info["tree"])
        if node["level"] == 1 and node["children"]
    ]
    return {
        "passages": passages,
        "tree": info["tree"],
        "levels": info["levels"],
        "leaves": info["leaves"],
        "llm_calls": info["llm_calls"],
        "llm_mode": llm_mode,
        "build_s": build_s,
        "node_counts": _node_counts_per_level(info["tree"]),
        "root_text": info["tree"]["text"],
        "mid_text": mid_nodes[0]["text"] if mid_nodes else "",
    }


## 4. Demo — print the tree artifact

`print_demo(exp)` prints the artifact from four angles: the tree headline (leaves / levels / LLM calls / build time, plus which LLM mode ran); the node count per level with its role (root / summaries / leaves); the root summary preview; one mid-level summary preview; then a takeaway explaining why the tree is a lossy but navigable index.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the tree artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 09-02 — Recursive tree build with LLM summarization")
    print(f"{exp['leaves']} leaves, {exp['levels']} levels, "
          f"{exp['llm_calls']} LLM calls in {exp['build_s']:.1f}s "
          f"[llm mode: {exp['llm_mode']}]")
    print("=" * 66)

    print(f"\n[1] Nodes per level:")
    for level, count in exp["node_counts"].items():
        role = "root" if level == exp["levels"] else (
            "leaves" if level == 0 else "summaries")
        print(f"    level {level}: {count:3d} node(s)  [{role}]")

    print(f"\n[2] Root summary (first {ROOT_PREVIEW} chars):")
    print(f"    {exp['root_text'][:ROOT_PREVIEW]}")

    print(f"\n[3] One mid-level summary (first {MID_PREVIEW} chars):")
    print(f"    {exp['mid_text'][:MID_PREVIEW] or '(no mid-level node)'}")

    print(f"\n[4] Takeaway")
    print("    The tree is a lossy but navigable index: level 0 keeps every")
    print("    chunk verbatim, and each higher level compresses a cluster")
    print("    into a few sentences. Retrieval walks the summaries and only")
    print("    expands the cluster that matches the question — lab 03")
    print("    compares that walk against scanning every chunk flat.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: the tree has at least 2 levels; the root exists with non-empty text; the number of leaves equals `N_PASSAGES`; every chunk is covered exactly once across the leaves; every non-leaf node carries non-empty summary text; and the build stayed within `llm_calls <= 40`. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    root = exp["tree"]
    covered = sorted(collect_chunk_ids(root))

    checks.append((f"tree has >= 2 levels (got {exp['levels']})",
                   exp["levels"] >= 2))
    checks.append(("root exists and has non-empty text",
                   bool(root.get("text", "").strip())))
    checks.append((f"number of leaves == {len(exp['passages'])} "
                   f"(got {exp['leaves']})",
                   exp["leaves"] == len(exp["passages"])))
    checks.append(("every chunk covered exactly once across leaves",
                   covered == list(range(len(exp["passages"])))))
    checks.append(("every non-leaf node has non-empty summary text",
                   all(node["text"].strip()
                       for node in _walk(root) if node["children"])))
    checks.append((f"llm_calls <= 40 (got {exp['llm_calls']})",
                   exp["llm_calls"] <= 40))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A minute or two: 24 passages embedded on CPU, a handful of GMM fits, and one `ChatOllama` call per internal node (typically 4-8 calls at temperature 0). If Ollama is down, the stub summarizer keeps the run green and the gate passes. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The tree itself: nodes per level with roles, the root summary (what the whole corpus head distilled to), and one mid-level summary — the compressible middle of the index lab 03 will walk.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet file is intact and Ollama is serving `qwen2.5-coder:7b`.


In [ ]:
verify_gate(exp)
